In [1]:
import pandas as pd

raw_data = pd.read_csv('../datasets/data_EUR_raw.csv', na_values=['-'])

Check the data type of each column in the raw data.

In [2]:
raw_data.dtypes

Publication                       int64
No. of obs                        int64
Temp                            float64
Prec                            float64
Clay                              int64
CEC                             float64
BD                              float64
pH                              float64
ST                              float64
WFPS                              int64
SOC                             float64
TN                              float64
C/N                             float64
NH4+-N                          float64
NO3_-N                          float64
MN                              float64
Crop type                        object
crop_class                       object
Sowing date                      object
Harvest date                     object
Irrigation (date)                object
Pesticides/Herbicides (date)     object
Fertilizer N type                object
fertilization_code               object
fertilization_class              object


Divide the variables into different groups

In [3]:
numeric_static_features = {
    'Clay': '粘土含量',
    'CEC': '阳离子交换容量',
    'BD': '土壤容重',
    'pH': 'pH',
    'SOC': '有机碳含量',
    'TN': '总氮含量',
    'C/N': '碳氮比',
}

numeric_dynamic_features = {
    'Temp': '温度',
    'Prec': '降水量',
    'ST': '土壤温度',
    'WFPS': '土壤含水量',
    'NH4+-N': '铵态氮',
    'NO3_-N': '硝态氮',
    'MN': '矿质氮含量（铵态+硝态氮）'
}

classification_static_features = {
    'crop_class': '作物类型',
}

fertilization_features = {
    'fertilization_class': '上次施肥类型',
    'Split N amount': '上次施肥量',
    'appl_class': '上次施肥方式',
    'ferdur': '该次测量距上次施肥的天数'
}

optional_features = {
    'NH4+-N': '铵态氮',
    'NO3_-N': '硝态氮',
    'MN': '矿质氮含量（铵态+硝态氮）'
}

auxiliary_variables = {
    'Sowing date': '播种日期',
    'Harvest date': '收获日期',
    'Fertilization date': '施肥日期',
    'Emission date': '排放日期(观测日期)',
}

group_variables = {
    'No. of obs': 'ID',
    'Publication': '文献编号',
    'control_group': '组号',
    'sowdur': '该次测量到播种之间的天数',
}

drop_variables = {
    'Crop type',
    'Irrigation (date)',
    'Pesticides/Herbicides (date)',
    'Fertilizer N type',
    'fertilization_code',
    'inhibitor',
    'Total N amount',
    'Appl.code',
    'SE',
    'Duration'
}

labels = {
    'Daily fluxes': 'N2O排放通量',
}

Make sure our group can cover all the columns.

In [4]:
len(set(raw_data.columns) - numeric_dynamic_features.keys() - numeric_static_features.keys() - fertilization_features.keys() - classification_static_features.keys() - group_variables.keys() - drop_variables - auxiliary_variables.keys() - labels.keys()) == 0

True

Remove redundant columns, adjust the column order and delete all samples from the fallow period.

In [5]:
processed_data = raw_data.drop(columns=drop_variables)
processed_data = processed_data[[
    *group_variables.keys(),
    *labels.keys(),
    *classification_static_features.keys(),
    *numeric_static_features.keys(),
    *fertilization_features.keys(),
    *numeric_dynamic_features.keys(),
]]

print('Before drop all samples of fallow period, total sample: ', len(processed_data))
fallow_mask = (processed_data['sowdur'] < 0) | (processed_data['control_group'] < 0)
print('Fallow samples: ', fallow_mask.sum())
processed_data = processed_data[~fallow_mask]
print('After delete all samples of fallow period, processed sample: ', len(processed_data))

Before drop all samples of fallow period, total sample:  48688
Fallow samples:  12578
After delete all samples of fallow period, processed sample:  36110


Check the statistical characteristics and missing proportion of numeric types.

In [6]:
numeric_columns = labels.keys() | numeric_static_features.keys() | numeric_dynamic_features.keys() | {'ferdur', 'Split N amount'}

numeric_df = processed_data[list(numeric_columns)]
stats = numeric_df.describe().transpose()

missing_ratio = numeric_df.isnull().sum() / len(numeric_df) * 100
missing_df = missing_ratio.to_frame(name='missing_ratio (%)')

numeric_summary = stats.join(missing_df).round(3)
numeric_summary

,count,mean,std,min,25%,50%,75%,max,missing_ratio (%)
BD,36110.0,1.358,0.162,0.30,1.30,1.400,1.400,1.80,0.000
Temp,36110.0,12.663,6.750,-16.30,8.10,12.600,17.200,31.40,0.000
Split N amount,25554.0,101.374,81.274,0.00,53.00,85.000,130.000,984.00,29.233
Prec,36110.0,2.456,6.819,0.00,0.00,0.000,1.800,262.50,0.000
SOC,36110.0,18.691,30.986,2.20,9.30,12.900,18.200,412.00,0.000
pH,36110.0,6.993,0.844,4.00,6.40,6.900,7.700,8.60,0.000
WFPS,36110.0,56.345,18.929,2.00,43.00,57.000,71.000,128.00,0.000
ferdur,36110.0,35.180,50.417,-1.00,-1.00,15.000,52.000,367.00,0.000
TN,24144.0,1.684,2.535,0.20,1.00,1.300,1.600,33.60,33.138
ST,36110.0,13.311,7.135,-11.90,8.30,13.100,18.000,51.10,0.000


Ensure that the `Split N amount` column only has missing values when `ferdur < 0`.

In [7]:
processed_data[(processed_data['Split N amount'].isnull()) & (processed_data['ferdur'] >= 0)]

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN


In [8]:
processed_data[(~processed_data['Split N amount'].isnull()) & (processed_data['ferdur'] < 0)]

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN
1765,1766,13,1,43,48.25,maize,41,32.0,1.3,7.6,...,40.0,deep,-1,24.9,0.0,20.1,42,NaN,NaN,NaN
1766,1767,13,2,43,180.63,maize,41,32.0,1.3,7.6,...,40.0,deep,-1,24.9,0.0,20.1,46,NaN,NaN,NaN
1774,1775,13,1,64,61.05,maize,41,32.0,1.3,7.6,...,40.0,deep,-1,24.6,0.2,23.8,52,NaN,NaN,NaN
1775,1776,13,2,64,70.12,maize,41,32.0,1.3,7.6,...,40.0,deep,-1,24.6,0.2,23.8,45,NaN,NaN,NaN
1887,1888,13,9,55,366.32,maize,41,32.0,1.3,7.6,...,140.0,deep,-1,22.0,2.6,20.2,57,NaN,NaN,NaN
7031,7304,29,13,17,16.46,vegetables,6,16.0,1.4,7.2,...,61.0,surface,-1,24.5,0.4,31.6,49,NaN,NaN,NaN
7032,7305,29,14,17,9.17,vegetables,6,16.0,1.4,7.2,...,26.0,deep,-1,24.5,0.4,31.1,52,NaN,NaN,NaN
7033,7306,29,15,17,19.37,vegetables,6,16.0,1.4,7.2,...,19.0,deep,-1,24.5,0.4,29.9,57,NaN,NaN,NaN
8644,8917,40,1,92,69.21,wheat,20,18.0,1.2,5.4,...,100.0,deep,-1,1.1,0.0,3.7,73,NaN,NaN,NaN
8645,8918,40,2,92,62.63,wheat,20,18.0,1.2,5.4,...,100.0,deep,-1,1.1,0.0,3.7,73,NaN,NaN,NaN


In [11]:
processed_data[processed_data['ferdur'] == 0]

,No. of obs,Publication,control_group,sowdur,Daily fluxes,crop_class,Clay,CEC,BD,pH,...,Split N amount,appl_class,ferdur,Temp,Prec,ST,WFPS,NH4+-N,NO3_-N,MN
8,9,1,3,20,1.48,fruit,28,16.0,1.4,7.6,...,14.0,deep,0,17.9,0.0,22.1,60,1.97,11.15,13.12
9,10,1,4,20,1.47,fruit,28,16.0,1.4,7.6,...,14.0,deep,0,17.9,0.0,22.1,60,1.97,11.31,13.28
10,11,1,5,20,5.30,fruit,28,16.0,1.4,7.6,...,14.0,deep,0,17.9,0.0,22.1,60,2.22,3.90,6.12
11,12,1,6,20,5.22,fruit,28,16.0,1.4,7.6,...,14.0,deep,0,17.9,0.0,22.1,60,2.26,4.15,6.40
20,21,1,3,27,5.42,fruit,28,16.0,1.4,7.6,...,14.0,deep,0,21.5,0.0,23.5,60,4.53,74.10,78.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45917,49058,217,2,31,2.46,maize,30,20.0,1.3,8.2,...,100.0,surface,0,19.6,0.0,21.0,55,NaN,NaN,120.36
46210,49351,218,4,0,11.19,barley,34,13.0,1.4,6.4,...,68.0,surface,0,2.5,0.0,2.8,83,68.98,6.91,75.88
46211,49352,218,5,0,16.99,barley,34,13.0,1.6,6.4,...,68.0,surface,0,2.5,0.0,2.8,97,4.22,0.53,4.74
46212,49353,218,6,0,24.25,barley,34,13.0,1.4,6.4,...,68.0,surface,0,2.5,0.0,2.8,87,80.72,11.58,92.30


Check the statistical characteristics and missing proportion of classification variables.

In [9]:
classification_columns = classification_static_features.keys() | (fertilization_features.keys() - {'ferdur', 'Split N amount'})

classification_df = processed_data[list(classification_columns)]
stats = classification_df.describe().transpose()

missing_ratio = classification_df.isnull().sum() / len(classification_df) * 100
missing_df = missing_ratio.to_frame(name='missing_ratio (%)')

classification_summary = stats.join(missing_df).round(3)
classification_summary


,count,unique,top,freq,missing_ratio (%)
crop_class,36110,11,wheat,11004,0.0
fertilization_class,36110,9,AN,8239,0.0
appl_class,36110,3,surface,22665,0.0
